**Notebook 03a:** Postcode to LSOA lookup. Looks up the 2011 LSOA and local authority district for each practice postcode using postcodes.io, and checks every LSOA against IMD 2019. Output: `postcode_lsoa_lookup.csv`, used by notebook 03.

In [4]:
import pandas as pd, requests, time

m = pd.read_csv('QOF_2024_25/MAPPING_NHS_GEOGRAPHIES_2425.csv')
wm = m[m['ICB_ODS_CODE'].isin(['QHL', 'QUA', 'QWU'])]
gp = pd.read_csv('GP_Practice_Postcode_Lookup.csv', header=None)[[0, 9]]
gp.columns = ['PRACTICE_CODE', 'postcode']
gp['postcode'] = gp['postcode'].str.strip().str.upper()
wm = wm.merge(gp, on='PRACTICE_CODE', how='left')

pcs = wm['postcode'].dropna().unique().tolist()
rows = []
for i in range(0, len(pcs), 100):
    r = requests.post('https://api.postcodes.io/postcodes',
                      json={'postcodes': pcs[i:i+100]}, timeout=60)
    r.raise_for_status()
    for item in r.json()['result']:
        res = item['result']
        if res:
            rows.append({'postcode': item['query'].upper(),
                         'lsoa11': res['codes']['lsoa11'],
                         'admin_district': res['admin_district']})
    time.sleep(0.5)

pc_lsoa = pd.DataFrame(rows).drop_duplicates('postcode')
pc_lsoa.to_csv('postcode_lsoa_lookup.csv', index=False)

imd = pd.read_csv('ONS_data/IMD_2019.csv')
print('Postcodes found:', len(pc_lsoa), 'of', len(pcs))
print('Share of LSOA codes matching IMD 2019:',
      pc_lsoa['lsoa11'].isin(imd['LSOA code (2011)']).mean())

Postcodes found: 409 of 409
Share of LSOA codes matching IMD 2019: 1.0


**Acknowledgement of AI assistance**

This notebook was written by Mifta Mahmood with assistance from Claude (Anthropic), a generative AI assistant, which helped to write and debug the code. All code was run, checked and verified by the author, and the outputs shown are from the author's own runs. The final decisions on data, methods and interpretation are the author's own.